# Wine Quality — pipeline draft with unfinished regression

**Trương Minh Hoàng · MSSV 25410008**

**ARCHIVED_SOURCE — source review, not a fresh benchmark.** Bản học tập lịch sử được giữ để giải thích quá trình làm bài. Output cũ và thông tin môi trường riêng đã được bỏ; xem README/AUDIT của dự án trước khi chạy. Historical calculations and known limitations are documented separately.

📌 WINE QUALITY – FULL MACHINE LEARNING PIPELINE (COLAB)

Dataset: UCI Wine Quality
Source: ucimlrepo (official UCI)
Tasks:

Classification (Low / High quality)

Regression (predict quality score)

So sánh mô hình

CÀI ĐẶT MÔI TRƯỜNG (Colab)

In [ ]:
!pip3 install -U ucimlrepo lightgbm xgboost

1️⃣ LOAD DATASET (ĐÚNG CHUẨN UCI)

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Fetch Wine Quality dataset (UCI ID = 186)
wine = fetch_ucirepo(id=186)

X = wine.data.features.copy()
y = wine.data.targets.copy()

print("Shape X:", X.shape)
print("Shape y:", y.shape)


Metadata (nên dùng trong báo cáo)

In [ ]:
print(wine.metadata.additional_info.summary)


2️⃣ EDA – KHÁM PHÁ DỮ LIỆU

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = X.copy()
df["quality"] = y["quality"]

# Thống kê mô tả
display(df.describe())

# Phân phối quality
plt.figure(figsize=(5,3))
df["quality"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribution of Wine Quality")
plt.xlabel("Quality"); plt.ylabel("Count")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()



📌 Nhận xét nên ghi trong báo cáo:

Alcohol tương quan dương mạnh với quality

Volatile acidity tương quan âm

Dataset không có missing values



3️⃣ ĐỊNH NGHĨA BÀI TOÁN

3.1 Classification

In [ ]:
# Low quality (≤5) vs High quality (≥6)
df["label"] = (df["quality"] >= 6).astype(int)


3.2 Regression

In [ ]:
# Dự đoán quality gốc (0–10)


4️⃣ TRAIN / TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

features = X.columns.tolist()

# Classification
X_cls = df[features]
y_cls = df["label"]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Regression
X_reg = df[features]
y_reg = df["quality"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)


5️⃣ TIỀN XỬ LÝ DỮ LIỆU

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_cls_scaled = scaler.fit_transform(X_train_cls)
X_test_cls_scaled  = scaler.transform(X_test_cls)

X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled  = scaler.transform(X_test_reg)


📌 Lý do:

Logistic Regression cần scale

Tree-based model không bắt buộc, nhưng scale không gây hại

6️⃣ MÔ HÌNH MACHINE LEARNING

6.1 Logistic Regression (Baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_cls_scaled, y_train_cls)

y_pred_logit = logit.predict(X_test_cls_scaled)
y_prob_logit = logit.predict_proba(X_test_cls_scaled)[:,1]

print("Logistic Accuracy:", accuracy_score(y_test_cls, y_pred_logit))
print("Logistic F1:", f1_score(y_test_cls, y_pred_logit))
print("Logistic AUC:", roc_auc_score(y_test_cls, y_prob_logit))


6.2 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train_cls, y_train_cls)

y_pred_dt = dt.predict(X_test_cls)

print("Decision Tree Accuracy:", accuracy_score(y_test_cls, y_pred_dt))
print("Decision Tree F1:", f1_score(y_test_cls, y_pred_dt))


6.3 LightGBM (Model mạnh)

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

lgbm.fit(X_train_cls, y_train_cls)

y_pred_lgbm = lgbm.predict(X_test_cls)
y_prob_lgbm = lgbm.predict_proba(X_test_cls)[:,1]

print("LightGBM Accuracy:", accuracy_score(y_test_cls, y_pred_lgbm))
print("LightGBM F1:", f1_score(y_test_cls, y_pred_lgbm))
print("LightGBM AUC:", roc_auc_score(y_test_cls, y_prob_lgbm))


6.4 XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train_cls, y_train_cls)

y_pred_xgb = xgb.predict(X_test_cls)
y_prob_xgb = xgb.predict_proba(X_test_cls)[:,1]

print("XGBoost Accuracy:", accuracy_score(y_test_cls, y_pred_xgb))
print("XGBoost F1:", f1_score(y_test_cls, y_pred_xgb))
print("XGBoost AUC:", roc_auc_score(y_test_cls, y_prob_xgb))


7️⃣ REGRESSION (THỰC NGHIỆM MỞ RỘNG)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)

print("RMSE:", rmse)
print("MAE :", mean_absolute_error(y_test_reg, y_pred_reg))
print("R2  :", r2_score(y_test_reg, y_pred_reg))


8️⃣ FEATURE IMPORTANCE

In [ ]:
importances = pd.Series(lgbm.feature_importances_, index=features)
importances.sort_values(ascending=False).plot(kind="bar", figsize=(8,4))
plt.title("Feature Importance (LightGBM)")
plt.show()


📌 Nhận xét:

Alcohol quan trọng nhất

Sulphates, volatile acidity ảnh hưởng lớn

9️⃣ KẾT LUẬN (NÊN GHI TRONG BÁO CÁO)

LightGBM và XGBoost cho hiệu suất tốt nhất

Logistic Regression là baseline phù hợp

Alcohol là đặc trưng quan trọng nhất

Regression giúp hiểu chi tiết hơn chất lượng rượu

Dataset sạch, không missing values